In [1]:
import time
import re
import os
import sys
from bs4 import BeautifulSoup
import requests
from pymongo.errors import ServerSelectionTimeoutError
from pymongo import MongoClient
import random
import json
import pandas as pd
from collections import Counter

client = MongoClient("localhost:27017", serverSelectionTimeoutMS=60)  # 链接mongodb
db_collection = client['专利数据']['人工智能专利_发明授权']

In [60]:
find_db = client['专利数据']['人工智能专利_发明公布']
line = 0
for item in db_collection.find():
    ann_no = item['申请号']
    for item2 in find_db.find({"_id":ann_no}):
        applicated_no = item2['申请公布号']
        db_collection.update_one({"_id":item['_id']},{"$set":{"申请公布号_correct":applicated_no}},upsert=True)
    line += 1
    if line%1000 == 0:
        print(line)

1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000
76000
77000
78000
79000
80000
81000
82000
83000
84000
85000
86000
87000
88000
89000
90000
91000
92000
93000
94000
95000
96000
97000
98000
99000
100000
101000
102000
103000
104000
105000
106000
107000
108000
109000
110000
111000
112000
113000
114000
115000
116000
117000
118000
119000
120000
121000
122000
123000
124000
125000
126000
127000


In [83]:
all_data = []
line = 47000
for item in db_collection.find().skip(line):
    try:
        applicated_no = item['申请公布号_correct']
        if pd.isna(applicated_no):
            change_no = item['patents_data']['directAssociations'].split("(")[0].strip()
            print(change_no)
            db_collection.update_one({"_id":item['_id']},{"$set":{"申请公布号_correct":change_no}},upsert=True)
    except Exception as e:
        print(e.args, item['_id'])
        change_no = item['patents_data']['applicationNumber'].strip()
        db_collection.update_one({"_id":item['_id']},{"$set":{"申请公布号_correct":change_no}},upsert=True)
        
    if line%1000  == 0:
        print(line)
    line+= 1

47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
('directAssociations',) 038161397
CN102017776A
CN102017779A
CN101808037A
CN102299846A
CN102111342A
CN102196377A
CN102404213A
CN102679935A
CN102640644A
CN102195889A
CN102413062A
CN102571591A
CN1072521A
CN101790740A
CN102037690A
CN102404181A
CN102571539A
CN102401874A
CN102314610A
CN102077521A
CN102480409A
CN102291298A
CN102470274A
CN102377671A
CN102413056A
CN102739527A
CN101502060A
CN102469016A
CN102347906A
CN102325083A
CN101630961A
CN102655477A
CN102474458A
CN102523167A
CN102647363A
CN101632268A
CN102282814A
CN101154281A
CN102571586A
CN102870382A
CN102210126A
CN102244889A
CN102025619A
CN102752207A
CN101997751A
CN102377636A
CN102469022A
CN102577264A
CN102195879A
CN102265698A
CN102356604A
CN102164089A
CN102769572A
CN102047332A
CN102624608A
CN102088390A
CN102724118A
CN102811154A
CN102598599A
CN102831478A
CN101505269A
CN102842318A
CN102695294A
CN102546356A
CN102497316A
CN102474448A
CN

In [52]:
# 获得发明授权的cited by数据

In [3]:
all_data = []
line = 0
for item in db_collection.find():
    try:
        _id,applicated_no,applicated_date, granted_no, granted_date,assignee = item['_id'],item['申请公布号_correct'],item['申请日'],item['授权公告号'],item['授权公告日'],item['专利权人']
        all_data.append({"_id":_id,"applicated_no":applicated_no,"applicated_date":applicated_date, "granted_no":granted_no, "granted_date":granted_date,"assignee":assignee})
    except Exception as e:
        print(e.args, _id)
        break
    if line%1000  == 0:
        print(line)
    line+= 1

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000
76000
77000
78000
79000
80000
81000
82000
83000
84000
85000
86000
87000
88000
89000
90000
91000
92000
93000
94000
95000
96000
97000
98000
99000
100000
101000
102000
103000
104000
105000
106000
107000
108000
109000
110000
111000
112000
113000
114000
115000
116000
117000
118000
119000
120000
121000
122000
123000
124000
125000
126000
127000


In [4]:
result_df = pd.DataFrame(all_data)
result_df

,_id,applicated_no,applicated_date,granted_no,granted_date,assignee
0,2015106287175,CN105260710A,2015.09.28,CN105260710B,2018.08.28,北京石油化工学院
1,2014800261936,CN105229629A,2014.03.05,CN105229629B,2018.10.16,谷歌有限责任公司
2,2014800161228,CN105229667A,2014.03.13,CN105229667B,2017.09.19,艾锐势科技公司
3,2015106733180,CN105260718A,2015.10.13,CN105260718B,2018.07.13,暨南大学
4,2015106003671,CN105260592A,2015.09.18,CN105260592B,2018.08.07,中国地质大学（武汉）
...,...,...,...,...,...,...
127700,2023114115777,CN117152131A,2023.10.30,CN117152131B,2023.12.29,深圳市希格莱特科技有限公司
127701,2023114217855,CN117152148A,2023.10.31,CN117152148B,2023.12.29,南通杰元纺织品有限公司
127702,2023114324255,CN117152157A,2023.10.31,CN117152157B,2023.12.29,南通三喜电子有限公司
127703,2023114450559,CN117173171A,2023.11.02,CN117173171B,2023.12.29,深圳市蓝钛科技有限公司


In [6]:
len(result_df['applicated_no'].dropna())

127705

In [8]:
result_df.to_excel("./results/人工智能_异构创新网络专利号_授权号_日期_专利权人.xlsx",index=None)

In [19]:
# 获得cited_by数据

In [9]:
citedby_data = []
line = 0
for item in db_collection.find():
    cited_by = item['cited by']
    
    for i in cited_by:
        i['_id'] = item['_id']
        i['granted_no']=item['授权公告号']
        i['granted_date'] = item['授权公告日']
        i['applicated_no'] = item['申请公布号_correct']
        i['applicated_date'] = item['申请日']
        citedby_data.append(i)
    if line%1000  == 0:
        print(line)
    line+= 1

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000
76000
77000
78000
79000
80000
81000
82000
83000
84000
85000
86000
87000
88000
89000
90000
91000
92000
93000
94000
95000
96000
97000
98000
99000
100000
101000
102000
103000
104000
105000
106000
107000
108000
109000
110000
111000
112000
113000
114000
115000
116000
117000
118000
119000
120000
121000
122000
123000
124000
125000
126000
127000


In [10]:
cb_df = pd.DataFrame(citedby_data)
cb_df

,Publication number,Priority date,Publication date,Assignee,Title,_id,granted_no,granted_date,applicated_no,applicated_date
0,CN105844614B (zh)*,2016-03-15,2019-08-23,广东工业大学,一种基于校对机器人角度的视觉指北方法,2015106287175,CN105260710B,2018.08.28,CN105260710A,2015.09.28
1,CN105868762A (zh)*,2016-03-28,2016-08-17,国网浙江省电力公司宁波供电公司,一种sf6表的读数方法及系统,2015106287175,CN105260710B,2018.08.28,CN105260710A,2015.09.28
2,CN106127204B (zh)*,2016-06-30,2019-08-09,华南理工大学,一种全卷积神经网络的多方向水表读数区域检测算法,2015106287175,CN105260710B,2018.08.28,CN105260710A,2015.09.28
3,CN106228161B (zh)*,2016-07-18,2019-04-19,电子科技大学,一种指针式表盘自动读数方法,2015106287175,CN105260710B,2018.08.28,CN105260710A,2015.09.28
4,CN107220645B (zh)*,2017-05-24,2021-02-26,河北省计量监督检测研究院,基于动态图像处理的水表识别方法,2015106287175,CN105260710B,2018.08.28,CN105260710A,2015.09.28
...,...,...,...,...,...,...,...,...,...,...
1310863,CN117333777B (zh)*,2023-12-01,2024-02-13,山东元明晴技术有限公司,一种坝体异常识别方法、装置及存储介质,202311181886X,CN116912257B,2023.12.29,CN116912257A,2023.09.14
1310864,CN117475806B (zh)*,2023-12-28,2024-03-29,深圳康荣电子有限公司,基于多维传感数据反馈的显示屏自适应响应方法及其装置,2023112434091,CN116985803B,2023.12.29,CN116985803A,2023.09.26
1310865,CN117827466A (zh)*,2024-03-04,2024-04-05,南京宁麒智能计算芯片研究院有限公司,一种多核芯片的动态温度管理方法及系统,2023113308378,CN117074925B,2023.12.29,CN117074925A,2023.10.16
1310866,CN117517932B (zh)*,2023-12-29,2024-03-12,南京邮电大学,一种芯粒间tsv测试电路及测试方法,2023114081889,CN117148117B,2023.12.29,CN117148117A,2023.10.27


In [36]:
# cb_df.to_csv("./results/人工智能_异构创新网络_original_cited_by.csv",index=None)

# 对cited_by数据进行处理

In [11]:
# 首先对Publication number进行
cb_df['Publication number_flag'] = cb_df['Publication number'].str.contains("^CN")

In [12]:
flag_1 = cb_df[cb_df['Publication number_flag']==True]
result1 = flag_1.reset_index(drop=True)

In [13]:
result1['Publication_number_pure'] = result1['Publication number'].apply(lambda x: x.split('(')[0].strip())

In [14]:
# 合并专利号数据
patents_no = result_df['applicated_no'].tolist()
patents_no.extend(result_df['granted_no'].tolist())
len(patents_no)

255410

In [15]:
result1['publication_flag'] = result1['Publication_number_pure'].isin(patents_no)

## 判断是否为授权专利引用同一申请专利，如果是则删除

In [16]:
flag_2 = result1[result1['publication_flag']==True]
flag_2 = flag_2.reset_index(drop=True)

In [17]:
flag_2.to_excel("./results/人工智能_异构创新网络_cited_by2.xlsx")

## 判断三、日期问题

In [18]:
flag_2['applicated_date'] = flag_2['applicated_date'].apply(lambda x:x.replace(".","-"))

In [19]:
flag_2['Date1'] = pd.to_datetime(flag_2['applicated_date'], format='%Y-%m-%d')
flag_2['Date2'] = pd.to_datetime(flag_2['Priority date'], format='%Y-%m-%d')

In [20]:
# 计算两列日期之间的差值
flag_2['Date_Diff'] = flag_2['Date2'] - flag_2['Date1']

In [21]:
# 判断日期差值是否为负，并创建一个新列来存储结果
flag_2['date_flag'] = flag_2['Date_Diff'].apply(lambda x: x.days > 0)

In [22]:
result_3 = flag_2[flag_2['date_flag'] == True]
result_3 = result_3.reset_index(drop=True)

In [23]:
result_3.to_excel("./results/人工智能_异构创新网络_cited_by3.xlsx")

# 获得citation数据

In [24]:
citation_data = []
line = 0
for item in db_collection.find():
    citations = item['patent citations']
    
    for i in citations:
        i['_id'] = item['_id']
        i['granted_no']=item['授权公告号']
        i['granted_date'] = item['授权公告日']
        i['applicated_no'] = item['申请公布号_correct']
        i['applicated_date'] = item['申请日']
        citation_data.append(i)
    if line%1000  == 0:
        print(line)
    line+= 1

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000
76000
77000
78000
79000
80000
81000
82000
83000
84000
85000
86000
87000
88000
89000
90000
91000
92000
93000
94000
95000
96000
97000
98000
99000
100000
101000
102000
103000
104000
105000
106000
107000
108000
109000
110000
111000
112000
113000
114000
115000
116000
117000
118000
119000
120000
121000
122000
123000
124000
125000
126000
127000


In [25]:
citation_df = pd.DataFrame(citation_data)

In [34]:
# citation_df.to_csv("./results/人工智能_异构创新网络_original_citation.csv",index=None)

# 对citation数据进行处理

In [26]:
# 首先对Publication number进行
citation_df['Publication number_flag'] = citation_df['Publication number'].str.contains("^CN")

In [27]:
flag_5 = citation_df[citation_df['Publication number_flag']==True]
result5 = flag_5.reset_index(drop=True)

In [28]:
result5['Publication_number_pure'] = result5['Publication number'].apply(lambda x: x.split('(')[0].strip())

In [29]:
# 合并专利号数据
patents_no = result_df['applicated_no'].tolist()
patents_no.extend(result_df['granted_no'].tolist())
len(patents_no)

255410

In [30]:
result5['publication_flag'] = result5['Publication_number_pure'].isin(patents_no)

## 判断是否为授权专利引用同一申请专利，如果是则删除

In [31]:
flag_6 = result5[result5['publication_flag']==True]
flag_6 = flag_6.reset_index(drop=True)

In [41]:
flag_6.to_excel("./results/人工智能_异构创新网络_citation2.xlsx")

## 判断三、日期问题

In [32]:
flag_6['applicated_date'] = flag_6['applicated_date'].apply(lambda x:x.replace(".","-"))

In [33]:
flag_6['Date1'] = pd.to_datetime(flag_6['applicated_date'], format='%Y-%m-%d')
flag_6['Date2'] = pd.to_datetime(flag_6['Priority date'], format='%Y-%m-%d')
# 计算两列日期之间的差值
flag_6['Date_Diff'] = flag_6['Date2'] - flag_6['Date1']

# 判断日期差值是否为负，并创建一个新列来存储结果
flag_6['date_flag'] = flag_6['Date_Diff'].apply(lambda x: x.days <= 0)

result_7 = flag_6[flag_6['date_flag'] == True]
result_7 = result_7.reset_index(drop=True)

In [34]:
result_7.to_excel("./results/人工智能_异构创新网络_citation3.xlsx")

# 处理temp_data数据

In [36]:
read_data = pd.read_excel("./results/temp_data.xlsx")
read_data

,citing_no,citing_date,citing_assignee,cited_no,cited_date,cited_assignee,citing_cited
0,CN105844614B,2016-03-15,广东工业大学,CN105260710B,2015-09-28,北京石油化工学院,CN105844614B_CN105260710B
1,CN112214368B,2020-10-23,英业达科技有限公司;英业达股份有限公司,CN105224430B,2015-10-29,艾德克斯电子（南京）有限公司,CN112214368B_CN105224430B
2,CN112910720B,2021-05-06,成都云智天下科技股份有限公司,CN105245362B,2015-09-14,河南工业大学; 西安电子科技大学,CN112910720B_CN105245362B
3,CN104608125B,2013-11-01,精工爱普生株式会社,CN105234938B,2011-07-11,精工爱普生株式会社,CN104608125B_CN105234938B
4,CN106239505B,2016-08-01,广东电网有限责任公司电力科学研究院,CN105234938B,2011-07-11,精工爱普生株式会社,CN106239505B_CN105234938B
...,...,...,...,...,...,...,...
154534,CN117152131B,2023-10-30,深圳市希格莱特科技有限公司,CN116862907A,2023-08-30,无锡市明通动力工业有限公司,CN117152131B_CN116862907A
154535,CN117152148B,2023-10-31,南通杰元纺织品有限公司,CN114757900A,2022-03-31,绍兴柯桥奇诺家纺用品有限公司,CN117152148B_CN114757900A
154536,CN117152148B,2023-10-31,南通杰元纺织品有限公司,CN115082466A,2022-08-22,倍利得电子科技（深圳）有限公司,CN117152148B_CN115082466A
154537,CN117152157B,2023-10-31,南通三喜电子有限公司,CN112862770A,2021-01-29,珠海迪沃航空工程有限公司,CN117152157B_CN112862770A


In [39]:
read_data['citing_assignee_pure'] = read_data['citing_assignee'].apply(lambda x:x.replace("全部","").replace("\n","").replace(" ","").strip())

In [40]:
read_data['cited_assignee_pure'] = read_data['cited_assignee'].apply(lambda x:x.replace("全部","").replace("\n","").replace(" ","").strip())

In [45]:
read_data['citing_count'] = read_data['citing_cited'].value_counts()

In [52]:
df = pd.DataFrame(read_data['citing_cited'].value_counts())
new_df = df.reset_index()
new_df

,citing_cited,count
0,CN113997292B_CN110355754B,2
1,CN111337895B_CN107462877B,2
2,CN113907707B_CN112353368B,2
3,CN112348806B_CN111783494B,2
4,CN109147940B_CN106815566B,2
...,...,...
153818,CN101382888B_CN100570556,1
153819,CN103823686B_CN100570556,1
153820,CN101751258B_CN100543674,1
153821,CN101937354B_CN100543674,1


In [53]:
final_df = pd.merge(read_data,new_df,on='citing_cited')
final_df

,citing_no,citing_date,citing_assignee,cited_no,cited_date,cited_assignee,citing_cited,citing_assignee_pure,cited_assignee_pure,citing_count,count
0,CN105844614B,2016-03-15,广东工业大学,CN105260710B,2015-09-28,北京石油化工学院,CN105844614B_CN105260710B,广东工业大学,北京石油化工学院,NaN,1
1,CN112214368B,2020-10-23,英业达科技有限公司;英业达股份有限公司,CN105224430B,2015-10-29,艾德克斯电子（南京）有限公司,CN112214368B_CN105224430B,英业达科技有限公司;英业达股份有限公司,艾德克斯电子（南京）有限公司,NaN,1
2,CN112910720B,2021-05-06,成都云智天下科技股份有限公司,CN105245362B,2015-09-14,河南工业大学; 西安电子科技大学,CN112910720B_CN105245362B,成都云智天下科技股份有限公司,河南工业大学; 西安电子科技大学,NaN,1
3,CN104608125B,2013-11-01,精工爱普生株式会社,CN105234938B,2011-07-11,精工爱普生株式会社,CN104608125B_CN105234938B,精工爱普生株式会社,精工爱普生株式会社,NaN,1
4,CN106239505B,2016-08-01,广东电网有限责任公司电力科学研究院,CN105234938B,2011-07-11,精工爱普生株式会社,CN106239505B_CN105234938B,广东电网有限责任公司电力科学研究院,精工爱普生株式会社,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...
154534,CN117152131B,2023-10-30,深圳市希格莱特科技有限公司,CN116862907A,2023-08-30,无锡市明通动力工业有限公司,CN117152131B_CN116862907A,深圳市希格莱特科技有限公司,无锡市明通动力工业有限公司,NaN,1
154535,CN117152148B,2023-10-31,南通杰元纺织品有限公司,CN114757900A,2022-03-31,绍兴柯桥奇诺家纺用品有限公司,CN117152148B_CN114757900A,南通杰元纺织品有限公司,绍兴柯桥奇诺家纺用品有限公司,NaN,1
154536,CN117152148B,2023-10-31,南通杰元纺织品有限公司,CN115082466A,2022-08-22,倍利得电子科技（深圳）有限公司,CN117152148B_CN115082466A,南通杰元纺织品有限公司,倍利得电子科技（深圳）有限公司,NaN,1
154537,CN117152157B,2023-10-31,南通三喜电子有限公司,CN112862770A,2021-01-29,珠海迪沃航空工程有限公司,CN117152157B_CN112862770A,南通三喜电子有限公司,珠海迪沃航空工程有限公司,NaN,1


In [54]:
final_df.to_excel("./results/result_temp_data.xlsx",index=None)

# 将曾用名改为现用名

In [63]:
read_data = pd.read_excel("./results/result_temp_data.xlsx",sheet_name=0)
cym_data = pd.read_excel("./results/result_temp_data.xlsx", sheet_name=7)

In [56]:
read_data

,citing_no,citing_date,citing_assignee,cited_no,cited_date,cited_assignee_pure,citing_cited
0,CN105844614B,2016-03-15,广东工业大学,CN105260710B,2015-09-28,北京石油化工学院,CN105844614B_CN105260710B
1,CN112214368B,2020-10-23,英业达科技有限公司;英业达股份有限公司,CN105224430B,2015-10-29,艾德克斯电子（南京）有限公司,CN112214368B_CN105224430B
2,CN112910720B,2021-05-06,成都云智天下科技股份有限公司,CN105245362B,2015-09-14,河南工业大学; 西安电子科技大学,CN112910720B_CN105245362B
3,CN104608125B,2013-11-01,精工爱普生株式会社,CN105234938B,2011-07-11,精工爱普生株式会社,CN104608125B_CN105234938B
4,CN106239505B,2016-08-01,广东电网有限责任公司电力科学研究院,CN105234938B,2011-07-11,精工爱普生株式会社,CN106239505B_CN105234938B
...,...,...,...,...,...,...,...
153818,CN113205509B,2021-05-24,山东省人工智能研究院;齐鲁工业大学,CN111709925B,2020-06-11,深圳科亚医疗科技有限公司,CN113205509B_CN111709925B
153819,CN115272165B,2022-05-10,推想医疗科技股份有限公司,CN111709925B,2020-06-11,深圳科亚医疗科技有限公司,CN115272165B_CN111709925B
153820,CN115222665B,2022-06-13,北京医准智能科技有限公司,CN111709925B,2020-06-11,深圳科亚医疗科技有限公司,CN115222665B_CN111709925B
153821,CN110596328B,2019-06-25,北京机械设备研究所,CN107449874B,2017-09-13,中国科学院地理科学与资源研究所,CN110596328B_CN107449874B


In [62]:
key_value = dict(zip(cym_data["citing_assignee"],cym_data['现用名']))
key_value

{'浙江科技学院': '浙江科技大学',
 '郑州轻工业学院': '郑州轻工业大学',
 '浙江大学城市学院': '浙大城市学院',
 '浙江大学宁波理工学院': '浙大宁波理工学院',
 '江西理工大学应用科学学院': '赣南科技学院',
 '温州大学瓯江学院': '温州理工学院',
 '华南理工大学广州学院': '广州城市理工学院',
 '中国计量学院': '中国计量大学',
 '广东技术师范学院': '广东技术师范大学',
 '广西师范学院': '南宁师范大学',
 '中州大学': '郑州工程技术学院',
 '安庆师范学院': '安庆师范大学',
 '四川理工学院': '四川轻化工大学',
 '南京化工职业技术学院': '南京科技职业学院',
 '西安财经学院': '西安财经大学',
 '淮海工学院': '江苏海洋大学',
 '苏州科技学院': '苏州科技大学',
 '大连民族学院': '大连民族大学',
 '河南中医学院': '河南中医药大学',
 '阜阳师范学院': '阜阳师范大学',
 '陕西理工学院': '陕西理工大学',
 '温州医学院眼视光研究院': '温州医科大学眼视光研究院',
 '广东商学院': '广东财经大学',
 '中国人民解放军西安通信学院': '中国人民解放军国防科技大学西安通信学院',
 '淮安信息职业技术学院': '江苏电子信息职业学院',
 '阿坝师范高等专科学校': '阿坝师范学院',
 '承德石油高等专科学校': '河北石油职业技术大学',
 '江西科技师范学院': '江西科技师范大学',
 '重庆工学院': '重庆理工大学',
 '成都信息工程学院': '成都信息工程大学',
 '南京邮电学院': '南京邮电大学',
 '中国人民解放军装甲兵技术学院': '中国人民解放军陆军装甲兵学院',
 '浙江海洋学院': '浙江海洋大学',
 '上海应用技术学院': '上海应用技术大学',
 '北京建筑工程学院': '北京建筑大学',
 '中国人民武装警察部队学院': '中国人民警察大学',
 '柳州师范高等专科学校': '广西科技师范学院',
 '广西工学院': '广西科技大学',
 '华北水利水电学院': '华北水利水电大学',
 '徐州师范大学': '江苏师范大学',
 '西安邮电学院': '西安邮电大学',
 '浙江林学院'

In [66]:
citing_assignee_list,cited_assignee_list = [],[]
for index in read_data.index:
    row_data = read_data.iloc[index]
    row_citing_assignee,row_cited_assignee = row_data['citing_assignee'], row_data['cited_assignee']
    citing_assignee_pure = []
    for key in row_citing_assignee.split(";"):
        if key.strip() in key_value:
            citing_assignee_pure.append(key_value[key.strip()])
        else:
            citing_assignee_pure.append(key.strip())

    citing_assignee_string = ';'.join(citing_assignee_pure)
    citing_assignee_list.append(citing_assignee_string)

    cited_assignee_pure = []
    for key2 in row_cited_assignee.split(";"):
        if key2.strip() in key_value:
            cited_assignee_pure.append(key_value[key2.strip()])
        else:
            cited_assignee_pure.append(key2.strip())
    cited_assignee_string = ';'.join(cited_assignee_pure)
    cited_assignee_list.append(cited_assignee_string)
    if index%100 == 0:
        print(index)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800
11900
12000
12100
12200
12300
12400
12500
12600
12700
12800
12900
13000
13100
13200
13300
13400
13500
13600
13700
13800
13900
14000
14100
14200
14300
14400
14500
14600
14700
14800
14900
15000
15100
15200
15300
15400
15500
15600
15700
15800
15900
16000
16100
16200
16300
16400
16500
16600
16700
16800
16900
17000
17100
17200
17300
17400
17500
17600
17700
17800
17900
18000
18100
18200
18300
18400
18

In [67]:
read_data['citing_assignee_pure'] = citing_assignee_list
read_data['cited_assignee_pure'] = cited_assignee_list

In [69]:
read_data.to_excel("./results/result_temp_data2.xlsx",index=None)

# 处理边数据

In [84]:
read_data

,citing_no,citing_date,citing_assignee,cited_no,cited_date,cited_assignee,citing_cited,citing_assignee_pure,cited_assignee_pure
0,CN105844614B,2016-03-15,广东工业大学,CN105260710B,2015-09-28,北京石油化工学院,CN105844614B_CN105260710B,广东工业大学,北京石油化工学院
1,CN112214368B,2020-10-23,英业达科技有限公司;英业达股份有限公司,CN105224430B,2015-10-29,艾德克斯电子（南京）有限公司,CN112214368B_CN105224430B,英业达科技有限公司;英业达股份有限公司,艾德克斯电子（南京）有限公司
2,CN112910720B,2021-05-06,成都云智天下科技股份有限公司,CN105245362B,2015-09-14,河南工业大学; 西安电子科技大学,CN112910720B_CN105245362B,成都云智天下科技股份有限公司,河南工业大学;西安电子科技大学
3,CN104608125B,2013-11-01,精工爱普生株式会社,CN105234938B,2011-07-11,精工爱普生株式会社,CN104608125B_CN105234938B,精工爱普生株式会社,精工爱普生株式会社
4,CN106239505B,2016-08-01,广东电网有限责任公司电力科学研究院,CN105234938B,2011-07-11,精工爱普生株式会社,CN106239505B_CN105234938B,广东电网有限责任公司电力科学研究院,精工爱普生株式会社
...,...,...,...,...,...,...,...,...,...
153818,CN113205509B,2021-05-24,山东省人工智能研究院;齐鲁工业大学,CN111709925B,2020-06-11,深圳科亚医疗科技有限公司,CN113205509B_CN111709925B,山东省人工智能研究院;齐鲁工业大学,深圳科亚医疗科技有限公司
153819,CN115272165B,2022-05-10,推想医疗科技股份有限公司,CN111709925B,2020-06-11,深圳科亚医疗科技有限公司,CN115272165B_CN111709925B,推想医疗科技股份有限公司,深圳科亚医疗科技有限公司
153820,CN115222665B,2022-06-13,北京医准智能科技有限公司,CN111709925B,2020-06-11,深圳科亚医疗科技有限公司,CN115222665B_CN111709925B,浙江医准智能科技有限公司,深圳科亚医疗科技有限公司
153821,CN110596328B,2019-06-25,北京机械设备研究所,CN107449874B,2017-09-13,中国科学院地理科学与资源研究所,CN110596328B_CN107449874B,北京机械设备研究所,中国科学院地理科学与资源研究所


In [85]:
citing_data = read_data[['citing_no','citing_assignee_pure']]
citing_data

,citing_no,citing_assignee_pure
0,CN105844614B,广东工业大学
1,CN112214368B,英业达科技有限公司;英业达股份有限公司
2,CN112910720B,成都云智天下科技股份有限公司
3,CN104608125B,精工爱普生株式会社
4,CN106239505B,广东电网有限责任公司电力科学研究院
...,...,...
153818,CN113205509B,山东省人工智能研究院;齐鲁工业大学
153819,CN115272165B,推想医疗科技股份有限公司
153820,CN115222665B,浙江医准智能科技有限公司
153821,CN110596328B,北京机械设备研究所


In [81]:
type_data = pd.read_excel("./results/result_temp_data.xlsx", sheet_name=8)


In [82]:
replace_dict = {'I': 1, 'E': 2, 'U': 3, 'R': 3}
type_data['Assignee_score'] = type_data['Assignee_type'].replace(replace_dict)
type_data

,citing_assignee,Assignee_type,Assignee_score
0,金阳娃,I,1
1,王伟,I,1
2,王华锋,I,1
3,居鹤华,I,1
4,崔洋,I,1
...,...,...,...
17565,广东美的楼宇科技有限公司,E,2
17566,江苏济中能源技术有限公司,E,2
17567,上海宝冶集团有限公司,E,2
17568,大田基业投资管理咨询（北京）有限公司,E,2


In [86]:
citing_temp = []
for index in citing_data.index:
    citing_no,assignees = citing_data.iloc[index]['citing_no'],citing_data.iloc[index]['citing_assignee_pure']
    split_assignees = assignees.split(";")
    combine_flag = True if len(split_assignees)>1 else False
    for i in range(len(split_assignees)):
        citing_temp.append({"citing":citing_no,"order":i+1,"citing_assignee":split_assignees[i],"is_combine":combine_flag})

In [87]:
citing_df = pd.DataFrame(citing_temp)
citing_df

,citing,order,citing_assignee,is_combine
0,CN105844614B,1,广东工业大学,False
1,CN112214368B,1,英业达科技有限公司,True
2,CN112214368B,2,英业达股份有限公司,True
3,CN112910720B,1,成都云智天下科技股份有限公司,False
4,CN104608125B,1,精工爱普生株式会社,False
...,...,...,...,...
168740,CN113205509B,2,齐鲁工业大学,True
168741,CN115272165B,1,推想医疗科技股份有限公司,False
168742,CN115222665B,1,浙江医准智能科技有限公司,False
168743,CN110596328B,1,北京机械设备研究所,False


In [88]:
citing_merge = pd.merge(citing_df,type_data,on='citing_assignee')


,citing,order,citing_assignee,is_combine,Assignee_type,Assignee_score
0,CN105844614B,1,广东工业大学,False,U,3
1,CN109492546B,1,广东工业大学,False,U,3
2,CN107443373B,1,广东工业大学,False,U,3
3,CN106041932B,1,广东工业大学,False,U,3
4,CN107123117B,1,广东工业大学,False,U,3
...,...,...,...,...,...,...
169262,CN107530878B,1,整形工具股份有限公司,False,E,2
169263,CN109035141B,1,上海皓桦科技股份有限公司,False,E,2
169264,CN110363075B,1,南京特殊教育师范学院,False,U,3
169265,CN113313701B,1,兰州智悦信息科技有限公司,False,E,2


In [90]:
cited_data = read_data[['cited_no','cited_assignee_pure']]
cited_data

,cited_no,cited_assignee_pure
0,CN105260710B,北京石油化工学院
1,CN105224430B,艾德克斯电子（南京）有限公司
2,CN105245362B,河南工业大学;西安电子科技大学
3,CN105234938B,精工爱普生株式会社
4,CN105234938B,精工爱普生株式会社
...,...,...
153818,CN111709925B,深圳科亚医疗科技有限公司
153819,CN111709925B,深圳科亚医疗科技有限公司
153820,CN111709925B,深圳科亚医疗科技有限公司
153821,CN107449874B,中国科学院地理科学与资源研究所


In [93]:
cited_temp = []
for index in cited_data.index:
    citing_no,assignees = cited_data.iloc[index]['cited_no'],cited_data.iloc[index]['cited_assignee_pure']
    split_assignees = assignees.split(";")
    combine_flag = True if len(split_assignees)>1 else False
    for i in range(len(split_assignees)):
        cited_temp.append({"citing":citing_no,"order":i+1,"cited_assignee":split_assignees[i],"is_combine":combine_flag})

In [96]:
cited_df = pd.DataFrame(cited_temp)
cited_df

,citing,order,cited_assignee,is_combine
0,CN105260710B,1,北京石油化工学院,False
1,CN105224430B,1,艾德克斯电子（南京）有限公司,False
2,CN105245362B,1,河南工业大学,True
3,CN105245362B,2,西安电子科技大学,True
4,CN105234938B,1,精工爱普生株式会社,False
...,...,...,...,...
168817,CN111709925B,1,深圳科亚医疗科技有限公司,False
168818,CN111709925B,1,深圳科亚医疗科技有限公司,False
168819,CN111709925B,1,深圳科亚医疗科技有限公司,False
168820,CN107449874B,1,中国科学院地理科学与资源研究所,False


In [98]:
type_data.columns = ['cited_assignee','Assignee_type','Assignee_score']

In [99]:

cited_merge = pd.merge(cited_df,type_data,on='cited_assignee')
cited_merge

,citing,order,cited_assignee,is_combine,Assignee_type,Assignee_score
0,CN105260710B,1,北京石油化工学院,False,U,3
1,CN105260709B,1,北京石油化工学院,False,U,3
2,CN110171000B,1,北京石油化工学院,True,U,3
3,CN109741306B,1,北京石油化工学院,False,U,3
4,CN109741306B,1,北京石油化工学院,False,U,3
...,...,...,...,...,...,...
169392,CN111398529B,1,株式会社而摩比特,False,E,2
169393,CN110139730B,1,卡耐基梅隆大学,False,U,3
169394,CN108471946B,1,珀迪迈垂克斯公司,False,E,2
169395,CN111366684B,1,上海兆莹自控设备有限公司,False,E,2


In [100]:
citing_merge.columns = ['citing','order','cited_assignee','is_combine','Assignee_type','Assignee_score']

In [101]:
concat_df = pd.concat([citing_merge,cited_merge])
concat_df

,citing,order,cited_assignee,is_combine,Assignee_type,Assignee_score
0,CN105844614B,1,广东工业大学,False,U,3
1,CN109492546B,1,广东工业大学,False,U,3
2,CN107443373B,1,广东工业大学,False,U,3
3,CN106041932B,1,广东工业大学,False,U,3
4,CN107123117B,1,广东工业大学,False,U,3
...,...,...,...,...,...,...
169392,CN111398529B,1,株式会社而摩比特,False,E,2
169393,CN110139730B,1,卡耐基梅隆大学,False,U,3
169394,CN108471946B,1,珀迪迈垂克斯公司,False,E,2
169395,CN111366684B,1,上海兆莹自控设备有限公司,False,E,2


In [104]:
concat_df = concat_df.reset_index(drop=True)
concat_df

,citing,order,cited_assignee,is_combine,Assignee_type,Assignee_score
0,CN105844614B,1,广东工业大学,False,U,3
1,CN109492546B,1,广东工业大学,False,U,3
2,CN107443373B,1,广东工业大学,False,U,3
3,CN106041932B,1,广东工业大学,False,U,3
4,CN107123117B,1,广东工业大学,False,U,3
...,...,...,...,...,...,...
338659,CN111398529B,1,株式会社而摩比特,False,E,2
338660,CN110139730B,1,卡耐基梅隆大学,False,U,3
338661,CN108471946B,1,珀迪迈垂克斯公司,False,E,2
338662,CN111366684B,1,上海兆莹自控设备有限公司,False,E,2


In [105]:
concat_df.to_excel("./results/edge_belong.xlsx",index=None)

In [151]:
# 
read_data = pd.read_excel("./results/edge_belong.xlsx")
read_data

,citing,citing_assignees,cited,cited_assignees,citing_year
0,CN105844614B,广东工业大学,CN105260710B,北京石油化工学院,2016
1,CN112214368B,英业达科技有限公司;英业达股份有限公司,CN105224430B,艾德克斯电子（南京）有限公司,2020
2,CN112910720B,成都云智天下科技股份有限公司,CN105245362B,河南工业大学;西安电子科技大学,2021
3,CN104608125B,精工爱普生株式会社,CN105234938B,精工爱普生株式会社,2013
4,CN106239505B,广东电网有限责任公司电力科学研究院,CN105234938B,精工爱普生株式会社,2016
...,...,...,...,...,...
98263,CN113205509B,山东省人工智能研究院;齐鲁工业大学,CN111709925B,深圳科亚医疗科技有限公司,2021
98264,CN115272165B,推想医疗科技股份有限公司,CN111709925B,深圳科亚医疗科技有限公司,2022
98265,CN115222665B,浙江医准智能科技有限公司,CN111709925B,深圳科亚医疗科技有限公司,2022
98266,CN110596328B,北京机械设备研究所,CN107449874B,中国科学院地理科学与资源研究所,2019


In [139]:
citing_data = read_data[['citing','citing_assignees']]
citing_temp = []
for index in citing_data.index:
    citing_no,assignees = citing_data.iloc[index]['citing'],citing_data.iloc[index]['citing_assignees']
    split_assignees = assignees.split(";")
    combine_flag = True if len(split_assignees)>1 else False
    for i in range(len(split_assignees)):
        citing_temp.append({"citing":citing_no,"order":i+1,"cited_assignee":split_assignees[i],"is_combine":combine_flag})

In [140]:
citing_df = pd.DataFrame(citing_temp)
citing_df

,citing,order,cited_assignee,is_combine
0,CN105844614B,1,广东工业大学,False
1,CN112214368B,1,英业达科技有限公司,True
2,CN112214368B,2,英业达股份有限公司,True
3,CN112910720B,1,成都云智天下科技股份有限公司,False
4,CN104608125B,1,精工爱普生株式会社,False
...,...,...,...,...
107776,CN113205509B,2,齐鲁工业大学,True
107777,CN115272165B,1,推想医疗科技股份有限公司,False
107778,CN115222665B,1,浙江医准智能科技有限公司,False
107779,CN110596328B,1,北京机械设备研究所,False


In [144]:
cited_data = read_data[['cited','cited_assignees']]
cited_temp = []
for index in cited_data.index:
    cited_no,assignees = cited_data.iloc[index]['cited'],cited_data.iloc[index]['cited_assignees']
    split_assignees = assignees.split(";")
    combine_flag = True if len(split_assignees)>1 else False
    for i in range(len(split_assignees)):
        cited_temp.append({"citing":cited_no,"order":i+1,"cited_assignee":split_assignees[i],"is_combine":combine_flag})

In [145]:
cited_df = pd.DataFrame(cited_temp)
cited_df

,citing,order,cited_assignee,is_combine
0,CN105260710B,1,北京石油化工学院,False
1,CN105224430B,1,艾德克斯电子（南京）有限公司,False
2,CN105245362B,1,河南工业大学,True
3,CN105245362B,2,西安电子科技大学,True
4,CN105234938B,1,精工爱普生株式会社,False
...,...,...,...,...
107806,CN111709925B,1,深圳科亚医疗科技有限公司,False
107807,CN111709925B,1,深圳科亚医疗科技有限公司,False
107808,CN111709925B,1,深圳科亚医疗科技有限公司,False
107809,CN107449874B,1,中国科学院地理科学与资源研究所,False


In [146]:
concat_df = pd.concat([citing_df,cited_df])
concat_df = concat_df.reset_index(drop=True)
concat_df

,citing,order,cited_assignee,is_combine
0,CN105844614B,1,广东工业大学,False
1,CN112214368B,1,英业达科技有限公司,True
2,CN112214368B,2,英业达股份有限公司,True
3,CN112910720B,1,成都云智天下科技股份有限公司,False
4,CN104608125B,1,精工爱普生株式会社,False
...,...,...,...,...
215587,CN111709925B,1,深圳科亚医疗科技有限公司,False
215588,CN111709925B,1,深圳科亚医疗科技有限公司,False
215589,CN111709925B,1,深圳科亚医疗科技有限公司,False
215590,CN107449874B,1,中国科学院地理科学与资源研究所,False


In [149]:

cited_merge = pd.merge(concat_df,type_data,on='cited_assignee',how='inner')
cited_merge

,citing,order,cited_assignee,is_combine,Assignee_type,Assignee_score
0,CN105844614B,1,广东工业大学,False,U,3
1,CN109492546B,1,广东工业大学,False,U,3
2,CN107443373B,1,广东工业大学,False,U,3
3,CN106041932B,1,广东工业大学,False,U,3
4,CN107123117B,1,广东工业大学,False,U,3
...,...,...,...,...,...,...
216277,CN110139730B,1,卡耐基梅隆大学,False,U,3
216278,CN108471946B,1,珀迪迈垂克斯公司,False,E,2
216279,CN110073404B,1,南坦生物组学有限责任公司,False,E,2
216280,CN110073404B,1,南坦生物组学有限责任公司,False,E,2


In [150]:
concat_df.to_excel("./results/edge_belong2.xlsx",index=None)

In [152]:
read_data

,citing,citing_assignees,cited,cited_assignees,citing_year
0,CN105844614B,广东工业大学,CN105260710B,北京石油化工学院,2016
1,CN112214368B,英业达科技有限公司;英业达股份有限公司,CN105224430B,艾德克斯电子（南京）有限公司,2020
2,CN112910720B,成都云智天下科技股份有限公司,CN105245362B,河南工业大学;西安电子科技大学,2021
3,CN104608125B,精工爱普生株式会社,CN105234938B,精工爱普生株式会社,2013
4,CN106239505B,广东电网有限责任公司电力科学研究院,CN105234938B,精工爱普生株式会社,2016
...,...,...,...,...,...
98263,CN113205509B,山东省人工智能研究院;齐鲁工业大学,CN111709925B,深圳科亚医疗科技有限公司,2021
98264,CN115272165B,推想医疗科技股份有限公司,CN111709925B,深圳科亚医疗科技有限公司,2022
98265,CN115222665B,浙江医准智能科技有限公司,CN111709925B,深圳科亚医疗科技有限公司,2022
98266,CN110596328B,北京机械设备研究所,CN107449874B,中国科学院地理科学与资源研究所,2019


In [154]:

is_self = []
for index in read_data.index:
    flag = False
    citing_assignees,cited_assignees = read_data.iloc[index]['citing_assignees'],read_data.iloc[index]['cited_assignees']
    split_assignees = citing_assignees.split(";")
    for assignee in split_assignees:
        if assignee.strip() in cited_assignees:
            flag = True
            break
    is_self.append(flag)

In [156]:
read_data['is_self'] = is_self
read_data.to_excel("./results/edge_belong1.xlsx",index=None)